# NFL Big Data Bowl 2027
## Title TBD: Which Combine tests translate to in-game success?

*Open Track write-up skeleton — draft, not a submission. Findings and figures will be filled in after the remaining approved tests are complete.*


## 1. Introduction

**Question.** Which NFL Scouting Combine test translates to an important in-game success metric, and does that answer change by position group?

We study four groups among the 2023–2025 rookie classes in the official competition data:

- **WR**
- **TE**
- **DB** — cornerbacks and safeties, treated as one group (CB + S, including listed `DB`)
- **DL** — one group (DE, DT, NT, and OLB who rush from a defensive line alignment)

Running backs are out of scope (only one RB is in this cohort). Offensive linemen are not part of this write-up.

**What this notebook has to do.** The scored entry must use the provided player tracking data and must link Combine tracking to regular-season performance. External public data can support that story; it cannot replace the official tracking.

This file is a runnable skeleton: path detection, a light data overview, and one small example plot per section so the notebook executes end to end on Kaggle and locally. The example plots use official timed Combine results and play-level fields in `player_play`. They are **not** the final analysis. Tracking-derived metrics (20-yard splits, get-off speed at 1.0s, Brakes / Re-Burst, and the full DL Burst path) will replace these placeholders once those sections are written.

A beginner walkthrough for creating a Kaggle notebook, attaching competition data, and attaching external nflverse files lives in [`KAGGLE_NOTEBOOK_GUIDE.md`](KAGGLE_NOTEBOOK_GUIDE.md).


## 2. Setup


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

plt.rcParams.update({
    "figure.figsize": (7.2, 4.6),
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
})

# Colorblind-friendly accents (Okabe–Ito).
COLOR = {
    "WR": "#0072B2",
    "TE": "#009E73",
    "DB": "#E69F00",
    "DL": "#D55E00",
    "Edge": "#D55E00",
    "Interior": "#0072B2",
    "other": "#333333",
}

# Official competition slug when the notebook is attached to this hackathon.
COMPETITION_SLUG = "nfl-big-data-bowl-2027"

# Position groups used throughout. RB / FB are excluded by design.
DB_POS = {"CB", "FS", "SS", "DB"}
DL_POS = {"DE", "DT", "NT", "OLB"}
EDGE_POS = {"DE", "OLB"}
INTERIOR_POS = {"DT", "NT"}
FOCUS_GROUPS = ("WR", "TE", "DB", "DL")

# Roster columns that are not part of this analysis and are dropped on load.
UNUSED_PLAYER_COLS = (
    "draft_round",
    "draft_pick_within_round",
    "draft_overall_pick",
)

MIN_PLAYS = 20  # minimum regular-season snaps for a player-level example point


def focus_group(nfl_position: str) -> str | float:
    if nfl_position == "WR":
        return "WR"
    if nfl_position == "TE":
        return "TE"
    if nfl_position in DB_POS:
        return "DB"
    if nfl_position in DL_POS:
        return "DL"
    return np.nan


def dl_subtype(nfl_position: str) -> str | float:
    if nfl_position in EDGE_POS:
        return "Edge"
    if nfl_position in INTERIOR_POS:
        return "Interior"
    return np.nan


def find_dir_with(filename: str, roots: list[Path]) -> Path | None:
    for root in roots:
        if not root.exists():
            continue
        direct = root / filename
        if direct.exists():
            return root
        for match in root.rglob(filename):
            return match.parent
    return None


def find_data_dir() -> Path:
    """Locate official competition CSVs on Kaggle or in the local ./data folder."""
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        preferred = kaggle_input / COMPETITION_SLUG
        candidates = []
        if preferred.exists():
            candidates.append(preferred)
        candidates.extend(sorted(p for p in kaggle_input.iterdir() if p.is_dir()))
        found = find_dir_with("players.csv", candidates)
        if found is not None:
            return found.resolve()
        raise FileNotFoundError(
            "Found /kaggle/input but no players.csv. "
            "Attach the NFL Big Data Bowl 2027 competition data under Input → Add Input."
        )

    here = Path.cwd()
    local_candidates = [
        here / "data",
        here.parent / "data",
        Path("/workspace/bdb2027/data"),
    ]
    found = find_dir_with("players.csv", local_candidates)
    if found is not None:
        return found.resolve()
    raise FileNotFoundError(
        "Could not find players.csv. Locally, run this notebook from the repo "
        "root so ./data is visible. On Kaggle, attach the competition data."
    )


def find_external_dir() -> Path | None:
    """Locate the public nflverse Next Gen Stats cache, if it was attached."""
    kaggle_input = Path("/kaggle/input")
    roots: list[Path] = []
    if kaggle_input.exists():
        roots.extend(sorted(p for p in kaggle_input.iterdir() if p.is_dir()))
    here = Path.cwd()
    roots.extend([
        here / "external",
        here.parent / "external",
        Path("/workspace/bdb2027/external"),
    ])
    return find_dir_with("ngs_receiving.parquet", roots)


def spearman(x: pd.Series, y: pd.Series) -> tuple[int, float, float]:
    d = pd.DataFrame({"x": x, "y": y}).dropna()
    n = int(len(d))
    if n < 8 or d["x"].nunique() < 2 or d["y"].nunique() < 2:
        return n, float("nan"), float("nan")
    rho, p = stats.spearmanr(d["x"], d["y"])
    return n, float(rho), float(p)


def annotate_rho(ax, n: int, rho: float, p: float, loc: str = "upper right") -> None:
    if n == 0 or not np.isfinite(rho):
        text = f"n={n}  (too few to correlate)"
    else:
        p_txt = f"{p:.1e}" if p < 1e-4 else f"{p:.3f}"
        text = f"n={n}   Spearman ρ = {rho:+.2f}   p = {p_txt}"
    va = "top" if "upper" in loc else "bottom"
    ha = "right" if "right" in loc else "left"
    x = 0.98 if ha == "right" else 0.02
    y = 0.96 if va == "top" else 0.04
    ax.text(x, y, text, transform=ax.transAxes, ha=ha, va=va, fontsize=9)


print("Setup complete.")


## 3. Data loading

On Kaggle, attached competition files are mounted under `/kaggle/input/<competition-slug>/`. This cell detects that slug by listing `/kaggle/input` and looking for `players.csv`. Locally it falls back to `./data`.


In [ ]:
DATA = find_data_dir()
EXTERNAL = find_external_dir()

print(f"Competition data: {DATA}")
print(f"External NGS cache: {EXTERNAL if EXTERNAL is not None else 'not attached (optional)'}")
print()
def fmt_size(n: int) -> str:
    if n < 1_000_000:
        return f"{n / 1_000:.1f} KB"
    return f"{n / 1_000_000:.1f} MB"

print("CSV files in the competition folder:")
for path in sorted(DATA.glob("*.csv")):
    print(f"  {path.name:32s}  {fmt_size(path.stat().st_size):>10s}")

# Small tables — fine to load in full. Draft columns are dropped immediately.
players = pd.read_csv(DATA / "players.csv")
players = players.drop(columns=[c for c in UNUSED_PLAYER_COLS if c in players.columns])
players["pos_group"] = players["nfl_position"].map(focus_group)
players["dl_subtype"] = players["nfl_position"].map(dl_subtype)

combine = pd.read_csv(DATA / "combine_results.csv")
games = pd.read_csv(DATA / "games.csv")
career = pd.read_csv(DATA / "player_career_successes.csv")

# Play-level outcomes used by the lightweight example plots.
# Full tracking files (combine_tracking, game_tracking_*) are left for later sections.
PLAY_COLS = [
    "game_id", "play_id", "nfl_id", "lined_up_position",
    "play_nullified_by_penalty", "pass_result",
    "target", "route_ran", "separation_at_pass_forward", "cushion",
    "player_get_off", "sack", "time_to_pressure", "time_to_qb_hurry",
    "unblocked_pressure",
]
player_play = pd.read_csv(DATA / "player_play.csv", usecols=PLAY_COLS)
player_play = player_play.merge(
    games[["game_id", "season", "season_type"]],
    on="game_id",
    how="left",
)
reg_clean = (
    (player_play["season_type"] == "REG")
    & (player_play["play_nullified_by_penalty"] == "N")
)
player_play = player_play.loc[reg_clean].copy()

print()
print(
    f"Loaded {len(players):,} players, "
    f"{len(combine):,} Combine result rows, "
    f"{len(games):,} games, and "
    f"{len(player_play):,} regular-season non-nullified player-plays."
)


## 4. Data overview

The competition data covers 510 rookies from the 2023–2025 Combine classes, with official Combine results, 10 Hz Combine tracking, and regular-season game tracking plus play-level outcomes. Game tracking is 2D (x/y only) and does not include the ball.

The counts below are roster sizes, not sample sizes for every test. A player only enters a later chart if he has the Combine number and enough regular-season snaps for that metric.


In [ ]:
focus = players[players["pos_group"].isin(FOCUS_GROUPS)].copy()
roster = (
    focus.groupby("pos_group", sort=False)
    .agg(
        n_players=("nfl_id", "nunique"),
        n_2023=("draft_year", lambda s: int((s == 2023).sum())),
        n_2024=("draft_year", lambda s: int((s == 2024).sum())),
        n_2025=("draft_year", lambda s: int((s == 2025).sum())),
    )
    .reindex(list(FOCUS_GROUPS))
)

# Combine completeness for the official timed tests we will lead with.
comb_focus = focus.merge(
    combine[["nfl_id", "ten_yd_split", "forty", "vertical", "broad_jump", "three_cone", "short_shuttle"]],
    on="nfl_id",
    how="left",
)
tests = {
    "10-yd split": "ten_yd_split",
    "40-yd": "forty",
    "Vertical": "vertical",
    "Broad jump": "broad_jump",
    "3-cone": "three_cone",
    "Short shuttle": "short_shuttle",
}
complete = []
for group in FOCUS_GROUPS:
    row = {"group": group, "n": int((comb_focus["pos_group"] == group).sum())}
    sub = comb_focus[comb_focus["pos_group"] == group]
    for label, col in tests.items():
        row[label] = int(sub[col].notna().sum())
    complete.append(row)
complete = pd.DataFrame(complete)

fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.4), constrained_layout=True)

ax = axes[0]
x = np.arange(len(roster))
ax.bar(x, roster["n_players"], color=[COLOR[g] for g in roster.index])
ax.set_xticks(x, roster.index)
ax.set_ylabel("Players in the competition roster")
ax.set_title("Focus-group roster counts")
for i, n in enumerate(roster["n_players"]):
    ax.text(i, n + 1.5, str(int(n)), ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, max(roster["n_players"]) * 1.15)

ax = axes[1]
labels = list(tests)
xs = np.arange(len(labels))
width = 0.2
for i, group in enumerate(FOCUS_GROUPS):
    row = complete[complete["group"] == group].iloc[0]
    vals = [100 * row[lab] / row["n"] if row["n"] else 0 for lab in labels]
    ax.bar(xs + (i - 1.5) * width, vals, width, label=group, color=COLOR[group])
ax.set_xticks(xs, labels, rotation=25, ha="right")
ax.set_ylabel("Share of group with a recorded test (%)")
ax.set_title("Official Combine test completeness")
ax.set_ylim(0, 100)
ax.legend(frameon=False, ncol=4, loc="upper right", fontsize=8)

fig.suptitle("Competition roster overview (focus groups only)", y=1.02)
plt.show()

print("Roster counts by class year (rows are position groups):")
print(roster.to_string())
print()
print("Players with a recorded official Combine test:")
print(complete.to_string(index=False))
print()
print(
    "Excluded from the analysis groups: "
    + ", ".join(
        f"{pos} n={n}"
        for pos, n in players.loc[~players["pos_group"].isin(FOCUS_GROUPS), "nfl_position"]
        .value_counts()
        .items()
    )
)


## 5. Defensive line — planned headline: 10-yard split → get-off → pressure

**TODO.** Replace this preview with the full DL Burst analysis from `dl_burst.py`: official 10-yard split versus tracking-derived get-off (speed 1.0s after snap), then get-off versus pressure and sack rate, with year controls, Holm correction, Edge vs Interior breakouts, and the Burst Score composites. The scored write-up must use Combine and game tracking for this path, not only the official `player_get_off` field.

The chart below is a lightweight stand-in so the skeleton runs without reading the multi-GB game-tracking files. It uses the official 10-yard split and the NFL's play-level `player_get_off` on regular-season standing pass-rush snaps (lined up EDGE or INTERIOR_LINE, at least 20 snaps).


In [ ]:
dl_ids = set(players.loc[players["pos_group"] == "DL", "nfl_id"])
rush = player_play[
    player_play["nfl_id"].isin(dl_ids)
    & player_play["pass_result"].notna()
    & player_play["lined_up_position"].isin({"EDGE", "INTERIOR_LINE"})
].copy()
rush["got_pressure"] = rush["time_to_pressure"].notna().astype(float)

dl_plays = (
    rush.groupby("nfl_id", sort=False)
    .agg(
        n_rush=("play_id", "size"),
        player_get_off=("player_get_off", "median"),
        pressure_rate=("got_pressure", "mean"),
    )
    .reset_index()
)
dl_plays = dl_plays[dl_plays["n_rush"] >= MIN_PLAYS]

dl = (
    players.loc[players["pos_group"] == "DL", ["nfl_id", "dl_subtype"]]
    .merge(combine[["nfl_id", "ten_yd_split"]], on="nfl_id", how="left")
    .merge(dl_plays, on="nfl_id", how="inner")
)
dl_plot = dl.dropna(subset=["ten_yd_split", "player_get_off"])

fig, ax = plt.subplots()
for subtype, sub in dl_plot.groupby("dl_subtype"):
    ax.scatter(
        sub["ten_yd_split"],
        sub["player_get_off"],
        c=COLOR[subtype],
        label=f"{subtype} (n={len(sub)})",
        alpha=0.85,
        edgecolors="white",
        linewidths=0.4,
        s=42,
    )
n, rho, p = spearman(dl_plot["ten_yd_split"], dl_plot["player_get_off"])
annotate_rho(ax, n, rho, p, loc="upper left")
ax.set_xlabel("Official Combine 10-yard split (sec, faster is left)")
ax.set_ylabel("Median NFL get-off time (sec, faster is down)")
ax.set_title("DL preview: official 10-yard split vs official get-off")
ax.legend(frameon=False, loc="lower right")
plt.show()

print(
    "Preview only — final headline uses tracking-derived speed at 1.0s after snap, "
    "then pressure rate. "
    f"Qualified DL with a 10-yard split and get-off: n={n}."
)


## 6. Defensive backs

**TODO.** Replace this preview after the approved DB tackling run: tackle rate per snap (solo / assist), tackles for loss, forced fumbles, tackle depth, and closing speed, plus supporting PFR missed-tackle percentage and yards per target allowed if we use that public source. Also bring in the existing coverage work (cushion, separation allowed, Sunday Speed / SSOE) and be clear about what does and does not survive multiple-testing correction.

The chart below is a lightweight preview: official 10-yard split versus mean cushion on regular-season coverage snaps (`cushion` recorded, at least 20 such snaps).


In [ ]:
db_ids = set(players.loc[players["pos_group"] == "DB", "nfl_id"])
cov = player_play[player_play["nfl_id"].isin(db_ids) & player_play["cushion"].notna()].copy()
db_plays = (
    cov.groupby("nfl_id", sort=False)
    .agg(n_cov=("play_id", "size"), mean_cushion=("cushion", "mean"))
    .reset_index()
)
db_plays = db_plays[db_plays["n_cov"] >= MIN_PLAYS]

db = (
    players.loc[players["pos_group"] == "DB", ["nfl_id"]]
    .merge(combine[["nfl_id", "ten_yd_split"]], on="nfl_id", how="left")
    .merge(db_plays, on="nfl_id", how="inner")
)
db_plot = db.dropna(subset=["ten_yd_split", "mean_cushion"])

fig, ax = plt.subplots()
ax.scatter(
    db_plot["ten_yd_split"],
    db_plot["mean_cushion"],
    c=COLOR["DB"],
    alpha=0.85,
    edgecolors="white",
    linewidths=0.4,
    s=42,
)
n, rho, p = spearman(db_plot["ten_yd_split"], db_plot["mean_cushion"])
annotate_rho(ax, n, rho, p)
ax.set_xlabel("Official Combine 10-yard split (sec, faster is left)")
ax.set_ylabel("Mean cushion on coverage snaps (yards)")
ax.set_title("DB preview: official 10-yard split vs mean cushion")
plt.show()

print(f"Preview only. Qualified DB with a 10-yard split and cushion sample: n={n}.")


## 7. Wide receivers

**TODO.** Replace this preview with the receiver story we keep: lead with the 20-yard split from Combine tracking, Sunday Speed / SSOE, separation and cushion from tracking, and the accel / decel work (Brakes, Re-Burst). Be explicit that official timed results are a preview and that production stats (yards, YAC) are supporting, not the success definition unless we adopt one we define ourselves.

The chart below is a lightweight preview: official 10-yard split versus mean separation at pass-forward on targeted regular-season throws (at least 20 targets with a recorded separation).


In [ ]:
def receiver_preview(group: str, color: str, title: str) -> int:
    ids = set(players.loc[players["pos_group"] == group, "nfl_id"])
    tgt = player_play[
        player_play["nfl_id"].isin(ids)
        & player_play["separation_at_pass_forward"].notna()
        & (player_play["target"] == 1)
    ].copy()
    agg = (
        tgt.groupby("nfl_id", sort=False)
        .agg(
            n_tgt=("play_id", "size"),
            mean_sep=("separation_at_pass_forward", "mean"),
        )
        .reset_index()
    )
    agg = agg[agg["n_tgt"] >= MIN_PLAYS]
    tab = (
        players.loc[players["pos_group"] == group, ["nfl_id"]]
        .merge(combine[["nfl_id", "ten_yd_split"]], on="nfl_id", how="left")
        .merge(agg, on="nfl_id", how="inner")
    )
    plot = tab.dropna(subset=["ten_yd_split", "mean_sep"])

    fig, ax = plt.subplots()
    ax.scatter(
        plot["ten_yd_split"],
        plot["mean_sep"],
        c=color,
        alpha=0.85,
        edgecolors="white",
        linewidths=0.4,
        s=42,
    )
    n, rho, p = spearman(plot["ten_yd_split"], plot["mean_sep"])
    annotate_rho(ax, n, rho, p)
    ax.set_xlabel("Official Combine 10-yard split (sec, faster is left)")
    ax.set_ylabel("Mean separation at pass-forward (yards)")
    ax.set_title(title)
    plt.show()
    return n


n_wr = receiver_preview(
    "WR",
    COLOR["WR"],
    "WR preview: official 10-yard split vs mean target separation",
)
print(f"Preview only. Qualified WR with a 10-yard split and separation sample: n={n_wr}.")


## 8. Tight ends

**TODO.** Replace this preview with the TE-specific results (often a smaller sample than WR). Do not pool TE with WR in the final write-up unless a robustness check is clearly labeled as such. Flag small samples in the text and on any player cards.

The chart below uses the same lightweight preview as the WR section.


In [ ]:
n_te = receiver_preview(
    "TE",
    COLOR["TE"],
    "TE preview: official 10-yard split vs mean target separation",
)
print(f"Preview only. Qualified TE with a 10-yard split and separation sample: n={n_te}.")


## 9. Next Gen Stats validation (external, public)

**TODO.** Bring in the official NGS receiving validation from `explore_ngs.py`: our tracking-based separation versus official NGS average separation, plus cushion, and the Combine-versus-NGS tests (including the WR 20-yard split vs cushion lean). Keep NGS as supporting evidence. The scored story still has to be built on the official Kaggle tracking data.

These tables are public Next Gen Stats receiving files republished by [nflverse](https://github.com/nflverse). They are allowed under the competition's external-data rule when they are public and free for every participant. Attribution is in `external/ATTRIBUTION.md`. This notebook does not use paid sources.

If the cache is not attached, the cell below prints a skip message and continues. On Kaggle, upload the nflverse files as a public dataset and attach it with Input → Add Input (see the guide).


In [ ]:
def load_ngs_receiving(folder: Path) -> pd.DataFrame | None:
    parquet = folder / "ngs_receiving.parquet"
    csv = folder / "ngs_receiving.csv"
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        return pd.read_csv(csv)
    return None


if EXTERNAL is None:
    print(
        "NGS cache not found. Attach a public Kaggle dataset that contains "
        "ngs_receiving.parquet (see KAGGLE_NOTEBOOK_GUIDE.md). Skipping this plot."
    )
else:
    ngs = load_ngs_receiving(EXTERNAL)
    if ngs is None:
        print(f"Found {EXTERNAL}, but it has no ngs_receiving.parquet or .csv. Skipping.")
    else:
        # nflverse NGS season totals are typically week == 0.
        season = ngs.copy()
        if "week" in season.columns:
            season = season[season["week"] == 0]
        if "season_type" in season.columns:
            season = season[season["season_type"] == "REG"]
        season = season[season["season"].between(2023, 2025)]

        rec_ids = set(players.loc[players["pos_group"].isin(["WR", "TE"]), "nfl_id"])
        ours = player_play[
            player_play["nfl_id"].isin(rec_ids)
            & player_play["separation_at_pass_forward"].notna()
            & (player_play["target"] == 1)
        ].copy()
        ours = (
            ours.groupby("nfl_id", sort=False)
            .agg(
                n_tgt=("play_id", "size"),
                our_separation=("separation_at_pass_forward", "mean"),
            )
            .reset_index()
        )
        ours = ours[ours["n_tgt"] >= MIN_PLAYS]
        ours = ours.merge(
            players[["nfl_id", "display_name", "pos_group"]],
            on="nfl_id",
            how="left",
        )

        # Lightweight name match for this skeleton. The full script also uses
        # a stable identifier map (see explore_ngs.py). Names are used only to
        # join and are not printed.
        ngs_career = (
            season.groupby("player_display_name", sort=False)
            .agg(
                ngs_separation=("avg_separation", "mean"),
                ngs_cushion=("avg_cushion", "mean"),
                ngs_targets=("targets", "sum"),
            )
            .reset_index()
        )
        matched = ours.merge(
            ngs_career,
            left_on="display_name",
            right_on="player_display_name",
            how="inner",
        )
        plot = matched.dropna(subset=["our_separation", "ngs_separation"])

        fig, ax = plt.subplots()
        for group, sub in plot.groupby("pos_group"):
            ax.scatter(
                sub["our_separation"],
                sub["ngs_separation"],
                c=COLOR[group],
                label=f"{group} (n={len(sub)})",
                alpha=0.85,
                edgecolors="white",
                linewidths=0.4,
                s=42,
            )
        n, rho, p = spearman(plot["our_separation"], plot["ngs_separation"])
        annotate_rho(ax, n, rho, p, loc="upper left")
        lo = min(plot["our_separation"].min(), plot["ngs_separation"].min())
        hi = max(plot["our_separation"].max(), plot["ngs_separation"].max())
        ax.plot([lo, hi], [lo, hi], color="#888888", linewidth=1, linestyle="--")
        ax.set_xlabel("Our mean separation at pass-forward (yards)")
        ax.set_ylabel("NGS average separation (yards)")
        ax.set_title("NGS validation preview: our separation vs official NGS")
        ax.legend(frameon=False, loc="lower right")
        plt.show()

        attr = EXTERNAL / "ATTRIBUTION.md"
        print(
            f"Name-matched WR/TE with both measurements: n={n}. "
            "This is a skeleton join; the full analysis uses the identifier map."
        )
        if attr.exists():
            print(f"Attribution file: {attr}")


## 10. Coach takeaways

**TODO.** After the remaining tests land, write one short, coach-readable takeaway per position group. Lead with what to trust at the Combine, then what not to over-read. Use the 0–100 composite scores where they are justified. Do not promise draft recommendations the data does not support.

Planned shape of this section:

- **DL.** Trust the 10-yard split as a get-off test; get-off is the path to pressure. State the Edge / Interior caveat in plain language.
- **DB.** Fill in after the tackling run. Separate coverage traits from tackling traits. Flag small samples.
- **WR.** Lead with the 20-yard split and the Sunday movement traits that actually hold. Say clearly when a speed test is not a production grade.
- **TE.** Same structure as WR, with a small-sample warning.


## 11. Limitations

**TODO.** Expand this list with any limits that show up in the remaining tests. Keep the tone factual.

Known limits to carry forward:

- The scored analysis has to use the provided tracking data and has to link Combine tracking to regular-season performance.
- Game tracking is 2D only (x/y). There is no ball in the game-tracking files, and get-off is measured from the rusher's own track, not from a quarterback or blocker track.
- The cohort is 2023–2025 rookies who have Combine data in this competition. Findings do not automatically extend to veterans or to players who skipped the Combine.
- Sample sizes shrink quickly once a minimum snap count is applied, especially for TE.
- Official timed tests in this skeleton are a preview. Several of our lead metrics (20-yard split from tracking, Re-Burst, Brakes, speed at 1.0s) come from the 10 Hz files and are not in this first pass.
- External Next Gen Stats / nflverse tables are supporting evidence only. They are public and free; paid sources are not used.
- Associations are not causal claims.


## 12. Appendix

**TODO.** Move longer methods, extra robustness tables, and code-heavy detail here so the main narrative can stay short. Official write-up word and figure limits live on the competition's Submission Requirements and Evaluation tabs — confirm those before the final draft.

Reproducibility notes for this skeleton:

- Competition data path is detected automatically (`/kaggle/input/...` on Kaggle, `./data` locally).
- External nflverse NGS files are optional and skipped if they are not attached.
- Example plots use only regular-season, non-nullified plays and a 20-snap / 20-target minimum.
- Running backs are excluded. The four focus groups are WR, TE, DB (CB + S), and DL.
- This notebook does not read `combine_tracking.csv` or the yearly `game_tracking_*.csv` files yet. Those files are required for the scored analysis and will be added in later sections.

Helper scripts in the private working repo (not required to run this skeleton): `dl_burst.py`, `explore_ngs.py`, `eda_accel_decel.py`, `explore_route_cod.py`, and the other analysis modules. Full numbers live in `RESULTS_LOG.md`.


In [ ]:
print("Skeleton run finished.")
print(f"  Data dir: {DATA}")
print(f"  External dir: {EXTERNAL}")
print(
    "  Next: fill each TODO with tracking-based results, then trim this notebook "
    "to the findings that survive."
)
